# Bayesian Optimization for Black-Box Functions (Submission 12)

This notebook implements  **"The Precision Harvest"** Bayesian Optimization pipeline configured for **Round 12 Queries** across all 8 independent black-box target functions.

### Core Strategic Configurations for Round 12:
* **High-Density Search Depth (300 Multi-Starts):** The global acquisition surface is scanned rigorously using **300 random multi-starts** per function via the L-BFGS-B local optimizer to guarantee global alignment without getting pinned in local non-convex valleys.
* **"The Precision Harvest" $\xi$-Mapping Architecture:**
  * **$\xi = 0.5000$ (Max Momentum Exploration):** Preserved for Channels 1, 3, 4, and 6 to exploit the positive discovery momentum found in recent rounds (especially Function 4) and shock remaining flat zones.
  * **$\xi = 0.0100$ (Transitioning to Refinement):** Applied to **Function 5**, systematically stepping down from the emergency shock value to precisely narrow down and harvest the $909.46$ peak region.
  * **$\xi = 0.0500$ (Stabilized Noisy Scope):** Slightly reduced for **Function 2** to lock in and stabilize recent optimization gains within the stochastic environment.
  * **$\xi = 0.0000$ (Absolute Pure Exploitation):** Dedicated strictly to **Function 7** and **Function 8** to run final precise peak climbing along local gradients.

In [ ]:
import numpy as np
import os
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, ConstantKernel
from scipy.optimize import minimize
from scipy.stats import norm

# Enforce inline visualization support
%matplotlib inline 

print("Environment initialized. Round 12 Precision Harvest pipeline online.")

## Definition of Object-Oriented Bayesian Optimizer

In [ ]:
class BayesianOptimizer:
    def __init__(self, is_noisy=False):
        """
        Initializes noise variance parameter (alpha).
        """
        # Alpha is maintained at 1e-2 for noisy Function 2, 1e-6 for others
        self.alpha = 1e-2 if is_noisy else 1e-6
        self.model = None
        self.X = None
        self.Y = None
        self.dim = None

    def load_and_fit(self, func_dir):
        """
        Loads cumulative datasets and trains the Gaussian Process Regressor.
        """
        X = np.load(os.path.join(func_dir, "initial_inputs.npy"))
        Y = np.load(os.path.join(func_dir, "initial_outputs.npy"))
        self.X, self.Y, self.dim = X, Y, X.shape[1]
        
        # Robust Matern 2.5 kernel for structural landscape tracking
        kernel = ConstantKernel(1.0) * Matern(
            length_scale=np.ones(self.dim), 
            nu=2.5
        )
        
        self.model = GaussianProcessRegressor(
            kernel=kernel, 
            alpha=self.alpha,
            normalize_y=True, 
            n_restarts_optimizer=50 
        )
        self.model.fit(self.X, self.Y)

    def expected_improvement(self, x, xi):
        """
        Computes Expected Improvement (EI) surfaces for specified jitter configurations.
        """
        x = x.reshape(-1, self.dim)
        mu, sigma = self.model.predict(x, return_std=True)
        mu_sample_opt = np.max(self.Y)

        with np.errstate(divide='warn'):
            imp = mu - mu_sample_opt - xi
            Z = imp / sigma
            ei = imp * norm.cdf(Z) + sigma * norm.pdf(Z)
            ei[sigma <= 0.0] = 0.0
        return ei

    def propose_next_point(self, xi):
        """
        Executes 300 random multi-starts to find the global acquisition optimum.
        """
        best_x = None
        max_ei = -1
        
        for _ in range(300):
            x0 = np.random.uniform(0.0, 1.0, self.dim)
            res = minimize(
                lambda x: -self.expected_improvement(x, xi),
                x0,
                bounds=[(0.0, 1.0)] * self.dim,
                method='L-BFGS-B'
            )
            if -res.fun > max_ei:
                max_ei = -res.fun
                best_x = res.x
        return best_x

## Sequential Execution Optimization Pipeline

In [ ]:
output_file = "submission_12_results.txt"

# xi_map Logic: "The Precision Harvest"
# F1, F3, F4, F6: Maintaining max exploration (0.5) to capture momentum.
# F2: Slightly reduced to 0.05 to stabilize gains on noisy functions.
# F5: Shifted down to 0.0100 from emergency shock to harvest the 909.46 peak.
# F7, F8: Pure exploitation (0.0000) for absolute local climbing.
xi_map = {
    1: 0.5000, 
    2: 0.0500, 
    3: 0.5000, 
    4: 0.5000, 
    5: 0.0100, 
    6: 0.5000, 
    7: 0.0000, 
    8: 0.0000  
}

print("Generating Submission 12: The Precision Harvest...")
print("-" * 55)

notebook_results = {}

for i in range(1, 8 + 1):
    func_dir = f"function_{i}"
    
    if not os.path.exists(func_dir):
        print(f"Warning: Functional directory context '{func_dir}' missing from workspace path.")
        continue
        
    optimizer = BayesianOptimizer(is_noisy=(i == 2))
    optimizer.load_and_fit(func_dir)
    
    current_xi = xi_map[i]
    next_point = optimizer.propose_next_point(xi=current_xi)
    formatted_point = "-".join([f"{val:.6f}" for val in next_point])
    
    notebook_results[f"Function {i}"] = (formatted_point, current_xi)
    print(f"Function {i}: {formatted_point} (xi={current_xi})")

# Write out query matrix parameters to target submission manifest
with open(output_file, "w") as f:
    for i in range(1, 9):
        key = f"Function {i}"
        if key in notebook_results:
            f.write(f"{key}: {notebook_results[key][0]}\n")

print("-" * 55)
print(f"Submission 12 successfully written onto file: {output_file}")

### Summary Evaluation Matrix

In [ ]:
print(f"| {'Target Channel':15} | {'Exploration (xi)':18} | {'Proposed Query Coordinates (Round 12)':55} |")
print(f"| {'-'*15} | {'-'*18} | {'-'*55} |")
for func, (coords, xi_val) in notebook_results.items():
    print(f"| {func:15} | {xi_val:<18} | {coords:55} |")